# PDEformer Finetuning: `viscoelastic_instability`

This notebook is a dataset-specific Well fine-tuning launcher in the style of `PDEformer_finetune_demo.ipynb`. It uses the equation registry in `src.data.well_equations` so PDEformer receives the `viscoelastic_instability` PDE DAG instead of the generic Well placeholder.

**Spatial dimensions:** 2

**Default selected fields:** `pressure, c_zz, velocity_x, velocity_y, C_xx, C_xy, C_yx, C_yy`


## Equation

$Re(u_t+u\cdot\nabla u)+\nabla p=\beta\Delta u+(1-\beta)\nabla\cdot T(C)$\n$C_t+u\cdot\nabla C+T(C)=C\nabla u+(\nabla u)^T C+\epsilon\Delta C$

**Registry note:** Dataset-specific equation DAG registered in `src.data.well_equations`.


In [3]:
from dataclasses import dataclass
from pathlib import Path

import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
if not (workspace_root / "src").exists():workspace_root = workspace_root.parent
sys.path.append(str(workspace_root))


from src.data.well_equations import get_well_equation_spec

WELL_DATASET = "viscoelastic_instability"
WELL_BASE_PATH = Path("/path/to/the_well")
TRAIN_SPLIT = "train"
TEST_SPLIT = "valid"
CHECKPOINT = Path("model-L.pt")
FIELD_INDICES = "0,1,2,3,4,5,6,7"
MAX_FIELDS = 8

spec = get_well_equation_spec(WELL_DATASET)
print(spec.pde_latex)


$Re(u_t+u\cdot\nabla u)+\nabla p=\beta\Delta u+(1-\beta)\nabla\cdot T(C)$\n$C_t+u\cdot\nabla C+T(C)=C\nabla u+(\nabla u)^T C+\epsilon\Delta C$


## Slurm Finetuning

Set `WELL_BASE_PATH` to the root containing the downloaded Well datasets. The Slurm wrapper writes a concrete YAML config into `slurm_logs/` and then runs `train.py`.


In [4]:
slurm_command = f'''sbatch --export=ALL,\
WELL_BASE_PATH={WELL_BASE_PATH},\
WELL_DATASET={WELL_DATASET},\
TRAIN_SPLIT={TRAIN_SPLIT},\
TEST_SPLIT={TEST_SPLIT},\
CHECKPOINT={CHECKPOINT},\
FIELD_INDICES={FIELD_INDICES},\
MAX_FIELDS={MAX_FIELDS} \
scripts/submit_the_well_finetune_slurm.sh'''
print(slurm_command)


sbatch --export=ALL,WELL_BASE_PATH=/path/to/the_well,WELL_DATASET=viscoelastic_instability,TRAIN_SPLIT=train,TEST_SPLIT=valid,CHECKPOINT=model-L.pt,FIELD_INDICES=0,1,2,3,4,5,6,7,MAX_FIELDS=8 scripts/submit_the_well_finetune_slurm.sh


## Local Smoke Test

Use a tiny sample count and `WANDB_MODE=offline` for a pipeline check before submitting a larger job.


In [5]:
local_command = f'''WELL_BASE_PATH={WELL_BASE_PATH} \
WELL_DATASET={WELL_DATASET} \
TRAIN_SAMPLES=1 TEST_SAMPLES=1 EPOCHS=0 \
FIELD_INDICES={FIELD_INDICES} MAX_FIELDS={MAX_FIELDS} \
WANDB_MODE=offline DEVICE_TARGET=CPU DEVICE_ID=0 \
bash scripts/submit_the_well_finetune_slurm.sh'''
print(local_command)


WELL_BASE_PATH=/path/to/the_well WELL_DATASET=viscoelastic_instability TRAIN_SAMPLES=1 TEST_SAMPLES=1 EPOCHS=0 FIELD_INDICES=0,1,2,3,4,5,6,7 MAX_FIELDS=8 WANDB_MODE=offline DEVICE_TARGET=CPU DEVICE_ID=0 bash scripts/submit_the_well_finetune_slurm.sh


## Equation-Aware Evaluation

The evaluator also defaults to `--pde-preset well_equation`, so the same registered equation DAG is used for raw or fine-tuned checkpoint checks.


In [ ]:
eval_command = f'''python scripts/evaluate_the_well.py \
  --config configs/inference/model-L.yaml \
  --checkpoint {CHECKPOINT} \
  --well-base-path {WELL_BASE_PATH} \
  --well-dataset {WELL_DATASET} \
  --split test \
  --num-samples 4 \
  --field-indices {FIELD_INDICES} \
  --pde-preset well_equation \
  --output exp/the_well/{WELL_DATASET}_test_equation_eval.json'''
print(eval_command)
